In [27]:
from spec2selfies.eval_utils import load_logs
from transformers import AutoTokenizer

from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import SimilarityMaps
from rdkit.Chem import rdMolDescriptors

import selfies as sf
from datasets import load_from_disk
from IPython.display import display
from collections import Counter
from tqdm import tqdm
import torch

In [ ]:
'''
Utilities
'''

In [21]:
valid_logs = ['text_ids', 'embeddings', 'spectras', 'bm_spectra', 'scores', 'initial_score', 'bm_text_ids', 'formulas', 'bm_formula']
LOGS_PATH = "/lustre/fsn1/projects/rech/aik/unh74gq/cache/vec2text/eval/jolly-clear-chow"
tokenizer = AutoTokenizer.from_pretrained("./data/saved_models/Inverter__30epochs__encoder_decoder_frozen/checkpoint-15411")
dataset = load_from_disk("./data/scaffold_split_merged_spectra/arrow/tokenized_datasets/with_formula/test")
#dataset = load_from_disk("./data/scaffold_split_merged_spectra/arrow/tokenized_datasets/test")

def decode_text_ids(text_ids, _id=0, step=0, beam=0): 
    if len(text_ids.shape) <= 2:
        decoded = tokenizer.decode(text_ids[_id,:].astype('uint8'), skip_special_tokens=True)
    else :
        decoded = tokenizer.decode(text_ids[_id, step, beam, :], skip_special_tokens=True)
    return decoded

def load(_type):
    assert _type in valid_logs
    path = f"{LOGS_PATH}/{_type}"
    return load_logs(path)

def draw_mol(selfies: str, label: str):
    smiles = sf.decoder(selfies)
    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, legend=label)
    display(img)

In [ ]:
'''
Mean sis per step calcul
'''

In [76]:
import torch

t = load('scores')
#t_bis = load('bm_text_ids')
#t_sc = load('scores')

print (t.shape)
t = torch.from_numpy(t)
valid = (t <= 1).all(dim=(1, 2))
print (valid.shape)
t = t[valid]
t = t.max(dim=2).values.mean(dim=0)
print (t)

(13996, 3, 15)
torch.Size([13996])
tensor([0.7275, 0.7788, 0.7788], dtype=torch.float16)


'\ni = 0\nfor k in range(t.shape[0]):\n    if t[k, 2] > 0.8:\n        i+=1\nprint (i)'

In [ ]:
'''
Mol VIZ
'''

In [22]:
count = 0
for index in range(len(dataset)):
    selfies = dataset['selfies'][index]
    if 'H' in selfies and count == 6:
        draw_mol(selfies, 'mol')
        print (selfies)
        smiles = sf.decoder(selfies)
        print (smiles)
        mol = Chem.MolFromSmiles(smiles)
        
        atom_counts = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
        atoms = set([atom.GetSymbol() for atom in mol.GetAtoms()])
        print (atoms)
        print(atom_counts)
        break
    elif 'H' in selfies:
        count+=1

100%|██████████| 13892/13892 [02:47<00:00, 83.13it/s]

1536


In [ ]:
'''
SANDBOX
'''

In [47]:
text_ids = dataset['input_ids'][0]
text_ids = [text_ids for _ in range (16)]
t = torch.tensor(text_ids)
def get_formula_vec(input_ids):
        selfies=tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        smiles = [sf.decoder(s) for s in selfies]
        mols = [Chem.MolFromSmiles(s) for s in smiles]
        
        atom_counts = [
            Counter(
                atom.GetSymbol() for atom in m.GetAtoms()
            )
            for m in mols
        ]
        
        formulas = [
            [ac.get(atom, 0) for atom in ATOM_LIST]
            for ac in atom_counts
        ]
        
        return torch.tensor(formulas)

form = get_formula_vec(t)

In [49]:
print (dataset)

Dataset({
    features: ['selfies', 'spectre', 'input_ids', 'attention_mask', 'labels', 'length', 'embedder_input_ids', 'embedder_attention_mask', 'formula'],
    num_rows: 13892
})
